# Grupo Bimbo Inventory Demand


<img src='https://webcdn.getmidas.com/uploads/2022/02/arz-talep-nedir.jpg'>


Bu çalışmada `Grupo Bimbo Inventory Demand` yarışması için ürün, rota ve müşteri bilgileri kullanılarak talep tahmini yapan bir başlangıç modeli oluşturulmuştur.


## Akış

1. Kütüphaneleri yükleme
2. Drive bağlama ve zip içeriğini tanımlama
3. Veri dosyalarını yükleme
4. Ön işleme
5. Özellik çıkarımı
6. Model kurma
7. RMSE değerlendirmesi
8. Test tahmini ve submission
9. Sonuç


## 1. Kütüphaneleri Yükleme


In [1]:
# Bu bölümde talep tahmini için gerekli veri hazırlama ve modelleme kütüphanelerini içe aktarıyoruz.


In [2]:
!pip install catboost -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


In [3]:
from google.colab import drive
import os
import zipfile
import tempfile
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import root_mean_squared_error
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
from catboost import CatBoostRegressor


## 2. Drive Bağlama ve Zip İçeriğini Tanımlama


In [4]:
# Bu bölümde yarışma zip dosyasını Drive üzerinden bağlıyor ve gerekli dosyaları buluyoruz.


In [5]:
drive.mount('/content/drive', force_remount=True)

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/grupo-bimbo-inventory-demand.zip'
base_data_dir = '/content/drive/MyDrive/Colab Data Dosyaları'

print('Veri klasörü mevcut mu?:', os.path.exists(base_data_dir))
if os.path.exists(base_data_dir):
    print('Klasördeki ilk dosyalar:', os.listdir(base_data_dir)[:20])

print('Zip dosyası:', zip_path)
print('Zip mevcut mu?:', os.path.exists(zip_path))
if not os.path.exists(zip_path):
    raise FileNotFoundError('Zip dosyası bulunamadı. Colab Data Dosyaları içindeki gerçek dosya adını kontrol et.')

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_members = zip_ref.namelist()

print('Zip içindeki ilk dosyalar:', zip_members[:20])

def find_zip_member(members, filename):
    for member in members:
        if member.endswith(filename):
            return member
    raise FileNotFoundError(f'{filename} zip içinde bulunamadi.')

train_member = find_zip_member(zip_members, 'train.csv.zip')
test_member = find_zip_member(zip_members, 'test.csv.zip')
sample_submission_member = find_zip_member(zip_members, 'sample_submission.csv.zip')

print('Train member:', train_member)
print('Test member:', test_member)
print('Sample submission member:', sample_submission_member)


Mounted at /content/drive
Veri klasörü mevcut mu?: True
Klasördeki ilk dosyalar: ['Skin_Data.zip', 'Date_Fruit_Dataset.zip', 'Grape_Disease_Detection_Dataset.zip', 'GermanTrafficSign.zip', 'Malaria Data.zip', 'Electronics_5.json.zip', 'Fashion Product Images.zip', 'YouTube Analytics Data.zip', 'E-Commerce Marketing & Sales Revenue Prediction.zip', 'Customer_Spending_Dataset.zip', 'Predict Customer Purchase Behavior Dataset.zip', 'E-commerce Customer Churn Dataset.zip', 'Predict Conversion in Digital Marketing Dataset.zip', 'Customer Segmentation.zip', 'Customer Personality Analysis.zip', 'Wholesale Customers Data.zip', 'MNIST Handwritten Digits.zip', 'Face Mask Detection Dataset.zip', 'Emotion Detection.zip', 'Twitter Tweets Sentiment Dataset.zip']
Zip dosyası: /content/drive/MyDrive/Colab Data Dosyaları/grupo-bimbo-inventory-demand.zip
Zip mevcut mu?: True
Zip içindeki ilk dosyalar: ['cliente_tabla.csv.zip', 'producto_tabla.csv.zip', 'sample_submission.csv.zip', 'test.csv.zip', 'town_

## 3. Veri Dosyalarını Yükleme


In [6]:
# Bu bölümde train ve test tablolarını yüklüyoruz. Veri büyük olduğu için ilk aşamada örneklem kullanıyoruz.


In [7]:
sample_train_rows = 600000
sample_test_rows = 200000


def read_csv_from_nested_zip(zip_path, member_name, nrows=None):
    with tempfile.TemporaryDirectory() as tmpdir:
        inner_zip_path = os.path.join(tmpdir, os.path.basename(member_name))

        with zipfile.ZipFile(zip_path, 'r') as outer_zip:
            with outer_zip.open(member_name) as src_file, open(inner_zip_path, 'wb') as dst_file:
                dst_file.write(src_file.read())

        with zipfile.ZipFile(inner_zip_path, 'r') as inner_zip:
            inner_members = inner_zip.namelist()
            csv_member = next((m for m in inner_members if m.endswith('.csv')), None)
            if csv_member is None:
                raise FileNotFoundError('İç zip içinde csv bulunamadı.')
            with inner_zip.open(csv_member) as f:
                return pd.read_csv(f, nrows=nrows)

train = read_csv_from_nested_zip(zip_path, train_member, nrows=sample_train_rows)
test = read_csv_from_nested_zip(zip_path, test_member, nrows=sample_test_rows)
sample_submission = read_csv_from_nested_zip(zip_path, sample_submission_member)

print('Train shape (sample):', train.shape)
print('Test shape (sample):', test.shape)
print('Sample submission shape:', sample_submission.shape)
train.head()


Train shape (sample): (600000, 11)
Test shape (sample): (200000, 7)
Sample submission shape: (6999251, 2)


,Semana,Agencia_ID,Canal_ID,Ruta_SAK,Cliente_ID,Producto_ID,Venta_uni_hoy,Venta_hoy,Dev_uni_proxima,Dev_proxima,Demanda_uni_equil
0,3,1110,7,3301,15766,1212,3,25.14,0,0.0,3
1,3,1110,7,3301,15766,1216,4,33.52,0,0.0,4
2,3,1110,7,3301,15766,1238,4,39.32,0,0.0,4
3,3,1110,7,3301,15766,1240,4,33.52,0,0.0,4
4,3,1110,7,3301,15766,1242,3,22.92,0,0.0,3


## 4. Ön İşleme


In [8]:
# Bu bölümde hedef değişkeni ayırıyor ve kullanılacak alanları düzenliyoruz.


In [9]:
train_df = train.copy()
test_df = test.copy()

for col in ['Semana', 'Agencia_ID', 'Canal_ID', 'Ruta_SAK', 'Cliente_ID', 'Producto_ID']:
    train_df[col] = pd.to_numeric(train_df[col], errors='coerce')
    test_df[col] = pd.to_numeric(test_df[col], errors='coerce')

print('Train null count:', train_df.isnull().sum().sum())
print('Test null count:', test_df.isnull().sum().sum())
train_df.head()


Train null count: 0
Test null count: 0


,Semana,Agencia_ID,Canal_ID,Ruta_SAK,Cliente_ID,Producto_ID,Venta_uni_hoy,Venta_hoy,Dev_uni_proxima,Dev_proxima,Demanda_uni_equil
0,3,1110,7,3301,15766,1212,3,25.14,0,0.0,3
1,3,1110,7,3301,15766,1216,4,33.52,0,0.0,4
2,3,1110,7,3301,15766,1238,4,39.32,0,0.0,4
3,3,1110,7,3301,15766,1240,4,33.52,0,0.0,4
4,3,1110,7,3301,15766,1242,3,22.92,0,0.0,3


## 5. Özellik Çıkarımı


In [10]:
# Bu bölümde model girdilerini hazırlıyor ve hedef değişkeni log dönüşümü ile daha dengeli hale getiriyoruz.


In [11]:
feature_columns = ['Semana', 'Agencia_ID', 'Canal_ID', 'Ruta_SAK', 'Cliente_ID', 'Producto_ID']

x = train_df[feature_columns].copy()
y = np.log1p(train_df['Demanda_uni_equil'].copy())
test_x = test_df[feature_columns].copy()

categorical_features = ['Agencia_ID', 'Canal_ID', 'Ruta_SAK', 'Cliente_ID', 'Producto_ID']
for col in categorical_features:
    x[col] = x[col].astype(str)
    test_x[col] = test_x[col].astype(str)

numerical_features = ['Semana']

x_train, x_valid, y_train, y_valid = train_test_split(
    x, y, test_size=0.2, random_state=42
)

cat_feature_indices = [x.columns.get_loc(col) for col in categorical_features]

print('x_train shape:', x_train.shape)
print('x_valid shape:', x_valid.shape)


x_train shape: (480000, 6)
x_valid shape: (120000, 6)


## 6. Model Kurma


In [12]:
# Bu bölümde temel ürün ve müşteri alanlarını kullanarak talep tahmini yapan başlangıç modelini kuruyoruz.


In [13]:
model = CatBoostRegressor(
    loss_function='RMSE',
    iterations=300,
    depth=8,
    learning_rate=0.1,
    random_state=42,
    verbose=0
)

model.fit(x_train, y_train, cat_features=cat_feature_indices)


CatBoostRegressor(depth=8, iterations=300, learning_rate=0.1, loss_function='RMSE', random_state=42, verbose=0)

## 7. RMSE Değerlendirmesi


In [14]:
# Bu bölümde modelin hata düzeyini log dönüşümlü hedef üzerinde RMSE ile ölçüyoruz.


In [15]:
valid_preds = model.predict(x_valid)
rmse = root_mean_squared_error(y_valid, valid_preds)
print('Validation RMSE:', round(rmse, 5))


Validation RMSE: 0.48289


## 8. Test Tahmini ve Submission


In [16]:
# Bu bölümde test verisi için talep tahminlerini üretip submission dosyasını oluşturuyoruz.


In [17]:
test_preds = model.predict(test_x)
final_preds = np.expm1(test_preds)
final_preds = np.clip(final_preds, a_min=0, a_max=None)

submission = pd.DataFrame({
    'id': test_df['id'].values,
    'Demanda_uni_equil': final_preds
})

print('Submission shape:', submission.shape)
submission.head()


Submission shape: (200000, 2)


,id,Demanda_uni_equil
0,0,6.348283
1,1,2.492139
2,2,3.374545
3,3,2.126468
4,4,6.348283


In [18]:
output_path = '/content/grupo_bimbo_submission.csv'
submission.to_csv(output_path, index=False)
print('Kaydedilen dosya:', output_path)


Kaydedilen dosya: /content/grupo_bimbo_submission.csv


## 9. Sonuç


Bu çalışmada Grupo Bimbo Inventory Demand yarışması için ürün, rota ve müşteri bilgileri kullanılarak talep tahmini yapan bir başlangıç modeli oluşturuldu. Elde edilen sonuçlara göre model validation verisi üzerinde 0.48289 RMSE değeri elde etti ve test verisi için talep tahminleri üretildi.